# ntkmirror — Onboarding Notebook

### Forward-pass fine-tuning for language models, end-to-end on a free GPU

This notebook gets you from zero to a working understanding of **[ntkmirror](https://github.com/leochlon/ntkmirror)** (Hassana Labs, Leon Chlon). By the end you will have, on a free Colab T4:

1. Fitted a **controller** that makes a small model better at a task — *without changing a single model weight*
2. **Composed** two controllers (maths + code) and seen them work together without interfering
3. Used a controller as **retrievable memory**
4. Understood *why* each of these works, not just *that* it does

**The one idea behind everything.** Normally, to make a model better at a task you *fine-tune* it — you change its weights, which is expensive and tends to make it worse at other things ("catastrophic forgetting"). ntkmirror does something different: it leaves the model's weights frozen and instead learns a tiny set of **dials** that gently turn individual internal channels up or down as information flows through:

$$h' = \exp(s)\,\cdot\,h$$

Here `h` is the model's internal signal on one channel, and `exp(s)` is a learned dial. If `s = 0` the dial does nothing (`exp(0) = 1`). Slightly positive turns the channel up; slightly negative turns it down. A **controller** is just a few thousand of these dials. That's the whole method — everything below is a consequence of it.

---
**Before you run anything:** set a GPU runtime. *Runtime → Change runtime type → T4 GPU → Save.* Then run cells top to bottom.

## 1 · Confirm you have a GPU

ntkmirror needs a GPU. This should print a table with `Tesla T4` (or similar). If it says no GPU is found, fix the runtime type (above) before continuing.

In [ ]:
!nvidia-smi

## 2 · Install ntkmirror

We clone the repo and install it in editable mode (`-e`), with the `[datasets]` extra so the composition section can pull GSM8K and MBPP. Takes a couple of minutes — most of it is downloading PyTorch-related wheels.

In [ ]:
import sys, subprocess, os

# Clean clone
os.chdir('/content')
subprocess.run(['rm', '-rf', 'ntkmirror'])
subprocess.run(['git', 'clone', '--quiet', 'https://github.com/leochlon/ntkmirror.git'], check=True)
os.chdir('/content/ntkmirror')

# Install dependencies (the package's own requirements + datasets extra)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[datasets]'], check=True)

# CRITICAL: add the source dir to the path directly, so the import below does NOT
# depend on the editable-install path cache being refreshed mid-session.
# This makes 'Run all' work the same as running cells one at a time.
src_path = '/content/ntkmirror/src'
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print('Install complete and source path registered.')

## 3 · Your first controller — the Python API

Before touching the command-line tools, let's do the smallest possible thing in Python so the moving parts are visible:

1. Load the base model (Qwen2.5-0.5B-Instruct — half a billion parameters, small enough for a free GPU).
2. Ask it a question **before** any controller — note the answer.
3. Fit a controller on a handful of examples.
4. Ask the **same** question again **with** the controller attached.

The training data here is five two-digit-addition examples. Tiny on purpose — the point is to *see the loop*, not to do serious training.

Watch for: the model's behaviour changing between 'before' and 'after', even though `model` itself is never modified — only a small controller is fit and attached.

In [ ]:
import sys, importlib, json
if '/content/ntkmirror/src' not in sys.path:
    sys.path.insert(0, '/content/ntkmirror/src')
importlib.invalidate_caches()

from transformers import AutoModelForCausalLM, AutoTokenizer
from ntkmirror import ForwardFineTuner, load_jsonl_examples

# --- tiny training set: five addition examples ---
examples = [
    {'prompt': 'Question: 14 + 27 = ?\nAnswer:', 'completion': ' 41'},
    {'prompt': 'Question: 36 + 18 = ?\nAnswer:', 'completion': ' 54'},
    {'prompt': 'Question: 52 + 29 = ?\nAnswer:', 'completion': ' 81'},
    {'prompt': 'Question: 73 + 16 = ?\nAnswer:', 'completion': ' 89'},
    {'prompt': 'Question: 45 + 38 = ?\nAnswer:', 'completion': ' 83'},
]
with open('/content/train_addition.jsonl', 'w') as f:
    for ex in examples:
        f.write(json.dumps(ex) + '\n')

# --- load the frozen base model ---
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype='auto').cuda()

test_prompt = 'Question: 47 + 36 = ?\nAnswer:'

# --- BEFORE: base model, no controller ---
print('=== BEFORE controller ===')
inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

# --- fit a controller (only the dials are trained; model stays frozen) ---
print('\n=== Fitting controller ===')
tuner = ForwardFineTuner(model, tokenizer, gates=2000)
stats = tuner.fit(load_jsonl_examples('/content/train_addition.jsonl'), steps=120)
print('fit stats:', {k: round(v, 4) if isinstance(v, float) else v for k, v in stats.items()})

# --- AFTER: same model + controller attached ---
print('\n=== AFTER controller ===')
print(tuner.generate(test_prompt, max_new_tokens=10))

### What you just saw

`tuner.fit(...)` did three things internally:
- **Selected** which channels to put dials on. It scores every channel by how much nudging it would reduce the loss, and keeps the top `gates` of them. (That's why a few thousand dials is enough — they're chosen surgically, not at random.)
- **Trained** only those dials with AdamW, for `steps` steps. The model's weights are frozen the whole time — look at `train_seconds`, it's tiny, because there are only a few thousand parameters to move.
- Left you a controller you can `save()`, `load()`, attach with `generate()`, or compose with others.

Note `selected_gates` in the stats — that's how many dials were actually used. Everything else in the model is untouched.

## 4 · The command-line tools

For real work you'll mostly use the CLI rather than the Python API — it handles fitting, evaluating, generating, composing, and memory. The three core verbs:

```bash
ntkmirror fit       --model ... --train train.jsonl --out controller.pt
ntkmirror eval      --model ... --controller controller.pt --eval eval.jsonl
ntkmirror generate  --model ... --controller controller.pt --prompt "..."
```

**Data format.** Examples are JSONL, one per line. The preferred schema is `prompt` + `completion` (the completion is the part the model is trained to produce):

```json
{"prompt": "Question: 14 + 27 = ?\nAnswer:", "completion": " 41"}
```

`instruction`/`response`, `question`/`answer`, and bare `text` are also accepted.

Let's run the built-in demo to see `fit` + `eval` from the command line:

In [ ]:
%cd /content/ntkmirror
# Small, fast version of the demo (512 gates, 40 steps) — finishes in a few minutes.
!GATES=512 STEPS=40 bash examples/run_demo.sh

### Reading the output

Two numbers matter, each reported `before` and `after`:

- **`nll`** (negative log-likelihood) = the model's error / how 'confused' it is. **Lower is better.**
- **`token_acc`** = how often it predicts the right next token. **Higher is better.**

You want `nll` to drop and `token_acc` to rise from `before` to `after`. The `fit` block also reports `select_seconds` (time to choose the dials) and `train_seconds` (time to train them) — both small, which is the point.

## 5 · Composition — the headline result

This is the property that makes ntkmirror more than a LoRA replacement. **Two controllers, each trained independently on a different task, can be added together — and both skills survive.**

Why this is hard: when you fine-tune (or LoRA) a model on task B after task A, it tends to forget A. Stacking adaptations usually means they fight. ntkmirror controllers don't, because (as we'll measure) the dials for different tasks point in nearly **independent directions** — so adding them doesn't create a conflict.

The script below:
1. Builds small GSM8K (maths) and MBPP (code) train/eval sets.
2. Fits a **maths** controller and, separately, a **code** controller.
3. **Composes** them (adds the dials) into one controller.
4. Evaluates all four — base, maths-only, code-only, composed — on **both** eval sets.

We run a **fast version first** (smaller, ~10–15 min) as a confidence check. The full-size run is in the next cell.

In [ ]:
%cd /content/ntkmirror
# Fast confidence check: smaller data, fewer gates, fewer steps.
!TRAIN_SIZE=24 EVAL_SIZE=16 GATES=1024 STEPS=80 bash scripts/run_disjoint_composition.sh

### Full-size composition run (~40–70 min)

This fits two 5000-gate controllers at 240 steps each, then runs all eight evaluations — the proper, quotable version. It's long: **keep the Colab tab active** so the free runtime doesn't disconnect. If you're short on time, the fast run above already demonstrates the effect; skip this and come back to it.

*Reference numbers from a representative full run (yours will be close, not identical — small data subsets and seeding vary):*

| Controller | Maths eval (nll ↓) | Code eval (nll ↓) |
|---|---|---|
| Base | 0.719 | 1.060 |
| Maths-only | **0.602** | 1.058 |
| Code-only | 0.738 | **0.794** |
| **Composed** | **0.616** | **0.912** |

The composed row improves on *both* tasks at once, staying close to each specialist. Gate cosine between the two controllers ≈ **0.02** (near-orthogonal) — that near-zero overlap is the mechanism behind clean composition.

## 5b · (Optional) Full-size composition — larger, cleaner numbers

The fast check above already demonstrates clean composition. This optional full run (5000 gates, 240 steps, 64 train / 32 eval) produces larger, smoother numbers — but it is **memory-heavy and long (~40–70 min)**, and on a free T4 it will **not** fit alongside the models already loaded by the cells above. So it is deliberately **not part of “Run all.”**

**To run it cleanly (recommended recipe):**
1. *Runtime → Restart session* (frees all GPU memory).
2. Run **only** the Setup cells (the GPU check and the install cell).
3. Then run the single command below in a fresh code cell:
   ```bash
   !cd /content/ntkmirror && bash scripts/run_disjoint_composition.sh
   ```
4. Keep the Colab tab active so the free runtime doesn't disconnect.

*Representative results from a full run (yours will be close, not identical — small data subsets and seeding vary):*

| Controller | Maths eval (nll ↓) | Code eval (nll ↓) |
|---|---|---|
| Base | 0.719 | 1.060 |
| Maths-only | **0.602** | 1.058 |
| Code-only | 0.738 | **0.794** |
| **Composed** | **0.616** | **0.912** |

The composed row improves on **both** tasks at once, staying close to each specialist — with gate cosine ≈ **0.02** between the two controllers (near-orthogonal), which is the mechanism behind clean composition. This is the property LoRA lacks.

### Reading the composition report

Beyond the eval table, the run writes a `composition_report.json` with the diagnostic that explains *why* composition works. Run the next cell to print the key fields.

In [ ]:
import json, pathlib
rep = pathlib.Path('runs/disjoint_composition/composition_report.json')
if rep.exists():
    obj = json.loads(rep.read_text())
    pair = obj['pairs'][0]
    print('gate_cosine    :', round(pair['gate_cosine'], 4), ' (0 = independent directions, 1 = identical)')
    print('overlap_gates  :', int(pair['overlap_gates']), 'of', int(pair['a_gates']), '(channels both controllers use)')
    print('jaccard        :', round(pair['jaccard'], 4), ' (fraction of the union that overlaps)')
    print()
    print('Interpretation: the two controllers often pick the SAME channels')
    print('(high overlap / jaccard) but turn them in NEARLY INDEPENDENT directions')
    print('(cosine near 0). Same real estate, non-conflicting adjustments — which')
    print('is why adding them preserves both skills.')
else:
    print('Run the composition cell above first.')

## 6 · Controllers as memory

If a controller is a small file that installs a behaviour on the forward pass, then a *library* of controllers is a form of memory: store many, retrieve the relevant one at query time, attach it, generate. No retraining, no stuffing context into the prompt.

The demo below:
1. Fits a small controller for two-digit carry-addition.
2. **Stores** it in a memory store with a text description and tags.
3. **Searches** the store with a natural-language query and retrieves the matching controller.
4. **Generates** an answer with the retrieved controller attached.

Watch the final answer — the model solves `47 + 36` correctly *because the right memory was retrieved and installed*, not because it was prompted with the method.

In [ ]:
%cd /content/ntkmirror
!bash examples/run_memory_demo.sh

### The memory verbs

The demo uses these under the hood — useful to know directly:

```bash
ntkmirror memory add      --store STORE --id NAME --controller ctrl.pt --text "..." --tags a,b
ntkmirror memory search   --store STORE --query "..." --top-k 2
ntkmirror memory generate --store STORE --query "..." --prompt "..." --top-k 1
```

Each stored item is small (the controller plus a little metadata), so a store can hold many behaviours and you only pay to attach the one(s) retrieved for a given query.

## 7 · Cheat-sheet: how to read any ntkmirror result

| Term | Plain meaning | Direction |
|---|---|---|
| `nll` | error / how confused the model is | lower = better |
| `token_acc` | how often the next token is right | higher = better |
| `selected_gates` | how many dials the controller uses | — |
| `max_log_gate` | hard cap on how far any dial can turn (default 0.05) | bounds each dial; the reason forgetting stays low |
| `gate_cosine` | how aligned two controllers' dials are | near 0 = compose cleanly |
| `overlap_gates` / `jaccard` | how many channels two controllers share | high overlap + low cosine = same channels, independent directions |
| `train_seconds` | time to train the dials | small, because only the dials move |

**The mental model in one line:** a controller is a small set of *bounded, surgically-placed volume dials* on a *frozen* model — fast to fit, gentle enough not to break existing skills, and stackable because different tasks' dials point in independent directions.

---

### Where to go next
- `docs/method.md` — the method and its failure modes
- `docs/composability.md` — why composition works
- `docs/persistent_memory.md` — the memory architecture
- `src/ntkmirror/controller.py` — the implementation; start at `ForwardFineTuner.fit`

Five questions worth being able to answer after reading the source:
1. Where exactly are the dials applied? (Which forward hook, on what tensor?)
2. How are the top-K channels selected before training?
3. What does the `tanh`-and-scale in the `s` property guarantee?
4. What is the loss the dials are trained against?
5. When two controllers compose, how are overlapping channels combined?